In [1]:
import pandas as pd
import numpy as np
import re
from src.config import BLD, RAW

In [2]:
RAW

PosixPath('/Users/hengdegao/python/thesis/data/original_data')

In [7]:
def process_asset_data(raw):
    columns = {
        "Stkcd": "Stkcd",
        "Accper": "year",
        "A001000000": "total_asset",
        "A003100000": "net_asset",
        "A001100000": "current_asset",
        "A002100000": "current_liability",
        "A002101000": "short_debt",
        "A002201000": "long_debt",
        "A002203000": "bonds_payable",
        "A002211000": "leasing",
    }
    df = pd.read_csv(raw / 'csmar' / '基本信息' / '资产负债表' / 'FS_Combas.csv',
                     usecols=columns.keys(), dtype={'Stkcd': str}).rename(columns=columns)

    df['year'] = pd.to_datetime(df['year'], errors='coerce').dt.year
    df = df.dropna(subset=['year'])  # 移除非法日期
    df = df.fillna(0)
    df = df[(df['Stkcd'] >= "000001") & (df['Stkcd'] <= "679999")]

    df['size'] = np.log(df['total_asset'].replace(0, np.nan))
    df['debt_ratio'] = (df['short_debt'] + df['long_debt'] + df['bonds_payable']) / df['total_asset']
    df['nwc'] = (df['current_asset'] - df['current_liability']) / df['total_asset']

    return df[['Stkcd', 'year', 'total_asset', 'net_asset', 'size', 'debt_ratio', 'nwc']]

process_asset_data(RAW)


,Stkcd,year,total_asset,net_asset,size,debt_ratio,nwc
0,000002,2012,3.788016e+11,6.382555e+10,26.660278,0.121352,0.271752
1,000002,2013,4.792053e+11,7.689598e+10,26.895395,0.102637,0.236067
2,000002,2014,5.084088e+11,8.816457e+10,26.954552,0.095459,0.234362
3,000002,2015,6.112956e+11,1.001835e+11,27.138846,0.089555,0.207694
4,000002,2016,8.306742e+11,1.134448e+11,27.445504,0.122901,0.170099
...,...,...,...,...,...,...,...
22136,603999,2015,1.917994e+09,1.590538e+09,21.374546,0.040667,0.719140
22137,603999,2016,1.937285e+09,1.643591e+09,21.384553,0.005162,0.612731
22138,603999,2017,1.983632e+09,1.690072e+09,21.408195,0.005041,0.515102
22139,603999,2018,2.002016e+09,1.679838e+09,21.417421,0.003996,0.486132


In [14]:
def process_income_data(raw):
    df = pd.read_csv(raw / 'csmar' / '基本信息' / '利润表' / 'FS_Comins.csv',
                     usecols=["Stkcd", "Accper", "B001100000", "B002000101", "B001210000"],
                     dtype={'Stkcd': str})
    df = df.rename(columns={
        "Accper": "year",
        "B001100000": "revenue",
        "B002000101": "net_profit",
        "B001210000": "manage_exp"
    })
    df['year'] = pd.to_datetime(df['year'], errors='coerce').dt.year
    df = df.dropna(subset=['year'])

    df = df.sort_values(by=["Stkcd", "year"])

    df["manage_exp_ratio"] = df["manage_exp"] / df["revenue"].replace(0, np.nan)

    df['revenue_growth'] = df.groupby('Stkcd')['revenue'].pct_change(fill_method=None)

    return df[['Stkcd', 'year', 'manage_exp_ratio', 'revenue', 'net_profit', 'revenue_growth']]

process_income_data(RAW)

,Stkcd,year,manage_exp_ratio,revenue,net_profit,revenue_growth
0,000002,2012,0.026963,1.031162e+11,1.255118e+10,NaN
1,000002,2013,0.022174,1.354188e+11,1.511855e+10,0.313263
2,000002,2014,0.026659,1.463880e+11,1.574545e+10,0.081002
3,000002,2015,0.024266,1.955491e+11,1.811941e+10,0.335828
4,000002,2016,0.028279,2.404772e+11,2.102261e+10,0.229754
...,...,...,...,...,...,...
23268,920819,2015,0.071912,3.333122e+09,1.823719e+08,NaN
23269,920819,2016,0.071421,4.557832e+09,2.029013e+08,0.367436
23270,920819,2017,0.066482,6.139669e+09,2.916319e+08,0.347059
23271,920819,2018,0.059789,6.232314e+09,4.434845e+08,0.015090


In [4]:
a= process_income_data(RAW)
a[a['Stkcd' ] == '002724']

,Stkcd,year,revenue,net_profit,revenue_growth
8518,002724,2014,1.056429e+09,1.716288e+08,NaN
8519,002724,2015,8.963748e+08,6.722589e+07,-0.151505
8520,002724,2016,9.248924e+08,1.112391e+08,0.031814
8521,002724,2017,1.096956e+09,1.519587e+08,0.186036
8522,002724,2018,1.253197e+09,1.899506e+08,0.142431
8523,002724,2019,1.494581e+09,2.573021e+08,0.192615


In [5]:
def process_capex_data(raw):
    df = pd.read_csv(raw / 'csmar/基本信息/现金流量表/FS_Comscfd.csv',
                     usecols=["Stkcd", "Accper", "C002006000", "C002010000"],
                     dtype={'Stkcd': str})
    df = df.rename(columns={
        "Accper": "year",
        "C002006000": "investment",
        "C002010000": "other_invest"
    })
    df['year'] = pd.to_datetime(df['year'], errors='coerce').dt.year
    df = df.dropna(subset=['year'])
    df = df.fillna(0)

    df['capex'] = np.log(df['investment'] + df['other_invest'] + 1)

    return df[['Stkcd', 'year', 'capex']]

process_capex_data(RAW)

,Stkcd,year,capex
0,000002,2012,19.420090
1,000002,2013,21.615015
2,000002,2014,22.669353
3,000002,2015,22.766708
4,000002,2016,21.986245
...,...,...,...
23268,920819,2015,20.128130
23269,920819,2016,19.613521
23270,920819,2017,19.649782
23271,920819,2018,19.859099


In [18]:
def clean_financial_data_full():
    df_asset = process_asset_data(RAW)
    df_income = process_income_data(RAW)
    df_capex = process_capex_data(RAW)
    # 合并资产和收入
    df = pd.merge(df_income, df_asset, on=["Stkcd", "year"], how="inner")
    df = pd.merge(df, df_capex, on=["Stkcd", "year"], how="inner")

    df["capex_ratio"] = np.exp(df["capex"] - df["size"])
    df["turnover_rate"] = df["revenue"] / df["total_asset"].replace(0, np.nan)
    # 计算 ROE 和 ROA
    df["roe"] = df["net_profit"] / df["net_asset"].replace(0, np.nan)
    df["roa"] = df["net_profit"] / df["total_asset"].replace(0, np.nan)

    df = df[[
        "Stkcd", "year", "revenue_growth", "size", "turnover_rate", 'manage_exp_ratio', "debt_ratio", "nwc", "roe", "roa", "capex_ratio"
    ]]

    # 保存
    output_file = BLD / 'financial_data_full_yearly.parquet'
    df.to_parquet(output_file, index=False)
    print(f"完整数据保存至：{output_file}")


if __name__ == "__main__":
    clean_financial_data_full()

完整数据保存至：/Users/hengdegao/python/thesis/data/processed/financial_data_full_yearly.parquet
